# M1.4: From weather to power — wind and solar capacity factors

## Objectives

At the end of this notebook you should be able to:

- define the **capacity factor** and use it to turn capacity (MW) into annual energy (MWh)
- explain why wind power grows with the **cube** of wind speed, and scale wind speeds to hub height
- turn wind speeds into power with a turbine **power curve**, and compute a capacity factor two independent ways
- calculate the **sun's position** from first principles and check it against pvlib
- estimate clear-sky solar irradiance and PV output, and explain why it is an upper bound
- download real hourly weather for a Croatian site from Open-Meteo (ERA5 reanalysis)

In M1.2 the wind and solar profiles were synthetic. This lesson builds the physics that turns weather into the `p_max_pu` time series PyPSA needs — so that, from now on, you know exactly what those numbers mean and where they come from.

In [1]:
import numpy as np
import pandas as pd
import pvlib
from scipy import integrate, stats

### Capacity factor

The **capacity factor** (CF) is the energy a plant actually delivers divided by what it would deliver running at full capacity all the time:

**CF = E / (P_nom × hours)**, so **E = P_nom × CF × hours**

It is the single most important number for comparing variable renewables. Capacity (MW) tells you the peak; capacity factor tells you the energy.

In [2]:
hours = 8760
for tech, p_nom_MW, cf in [("wind", 1000, 0.25), ("solar", 1000, 0.15), ("nuclear", 1000, 0.90)]:
    print(f"{tech:8s} {p_nom_MW} MW x CF {cf:.2f} -> {p_nom_MW * cf * hours / 1e6:.2f} TWh per year")

wind     1000 MW x CF 0.25 -> 2.19 TWh per year
solar    1000 MW x CF 0.15 -> 1.31 TWh per year
nuclear  1000 MW x CF 0.90 -> 7.88 TWh per year


The same 1,000 MW of capacity delivers six times more energy as a baseload nuclear plant than as solar. That is why "Croatia installed X MW of solar" says little until you know the capacity factor — and why the rest of this lesson is about computing it from weather.

### Wind: the cube law

Moving air carries kinetic energy. The power flowing through a rotor of swept area A is

**P = ½ ρ A v³**

with air density ρ ≈ 1.225 kg/m³ and wind speed v. The **cube** is the key fact: double the wind speed and the power rises eightfold. (A real turbine can extract at most 16/27 ≈ 59 % of it — the Betz limit — and in practice about 40–50 %.)

In [3]:
rho, rotor_diameter_m = 1.225, 120
area = np.pi * (rotor_diameter_m / 2) ** 2
for v in [6, 9, 12]:
    print(f"v = {v:2d} m/s -> power in the wind {0.5 * rho * area * v**3 / 1e6:5.2f} MW")

v =  6 m/s -> power in the wind  1.50 MW
v =  9 m/s -> power in the wind  5.05 MW
v = 12 m/s -> power in the wind 11.97 MW


From 6 to 12 m/s the available power grows by 2³ = 8. Small differences in average wind speed therefore make large differences in energy — remember this when you see the capacity factors below.

**Hub height.** Wind speed grows with height above ground, because the ground slows the air down. Weather data are often given at 10 m or 100 m, while modern hubs sit at 100–150 m. A common approximation is the **power law** v(h) = v_ref · (h / h_ref)^α with α ≈ 1/7 over open terrain (α depends strongly on terrain — treat it as an assumption):

In [4]:
v_10m = 5.0
for h in [50, 100, 150]:
    print(f"{v_10m} m/s at 10 m -> {v_10m * (h / 10) ** (1 / 7):.2f} m/s at {h} m")

5.0 m/s at 10 m -> 6.29 m/s at 50 m
5.0 m/s at 10 m -> 6.95 m/s at 100 m
5.0 m/s at 10 m -> 7.36 m/s at 150 m


### The turbine power curve

A turbine does not follow the cube law at all speeds. Its **power curve** has four regions:

| Wind speed | Output |
|---|---|
| below **cut-in** (~3 m/s) | 0 — not enough wind to turn the rotor usefully |
| cut-in to **rated** (~12 m/s) | rises roughly with v³ |
| rated to **cut-out** (~25 m/s) | flat at rated power — the blades pitch to spill excess wind |
| above cut-out | 0 — the turbine shuts down to protect itself |

The function below is a **generic** per-unit curve (output as a fraction of rated power), not the curve of a specific turbine model. Real curves come from manufacturers.

In [5]:
def power_curve(v, v_cut_in=3.0, v_rated=12.0, v_cut_out=25.0):
    """Generic per-unit turbine output for wind speed(s) v in m/s."""
    v = np.asarray(v, dtype=float)
    rising = (v ** 3 - v_cut_in ** 3) / (v_rated ** 3 - v_cut_in ** 3)
    out = np.where((v >= v_cut_in) & (v < v_rated), rising, 0.0)
    return np.where((v >= v_rated) & (v <= v_cut_out), 1.0, out)

pd.Series(power_curve([2, 3, 6, 9, 12, 20, 25, 26]).round(3), index=[2, 3, 6, 9, 12, 20, 25, 26]).rename_axis("wind speed m/s").rename("output p.u.")

wind speed m/s
2     0.000
3     0.000
6     0.111
9     0.413
12    1.000
20    1.000
25    1.000
26    0.000
Name: output p.u., dtype: float64

### Capacity factor from a wind-speed distribution

Wind speeds at a site are usually described by a **Weibull distribution** with shape k (≈ 2 for many sites) and scale c (m/s). The capacity factor is the power curve averaged over that distribution:

**CF = ∫ P(v) · f(v) dv**

Compute it two independent ways — numerical integration, and simulating 200,000 hours — and check that they agree. Two methods agreeing is a validation; one method alone is just a number.

In [6]:
def weibull_capacity_factor(c, k=2.0):
    """CF of the generic turbine for Weibull-distributed wind speeds (scale c in m/s)."""
    integrand = lambda v: power_curve(v) * stats.weibull_min.pdf(v, k, scale=c)
    value, _ = integrate.quad(integrand, 0, 30, points=[3, 12, 25], limit=200)
    return value

rng = np.random.default_rng(0)
rows = []
for c in [6, 7, 8, 9]:
    speeds = stats.weibull_min.rvs(2.0, scale=c, size=200_000, random_state=rng)
    rows.append({"scale_c_m_per_s": c, "mean_speed_m_per_s": round(speeds.mean(), 2),
                 "CF_integration": round(weibull_capacity_factor(c), 3),
                 "CF_simulation": round(power_curve(speeds).mean(), 3)})
cf_table = pd.DataFrame(rows)
assert np.allclose(cf_table.CF_integration, cf_table.CF_simulation, atol=0.005)
cf_table

   scale_c_m_per_s  mean_speed_m_per_s  CF_integration  CF_simulation
0                6                5.31           0.147          0.147
1                7                6.21           0.222          0.223
2                8                7.10           0.301          0.301
3                9                7.99           0.376          0.377

The two methods agree to within a few tenths of a percentage point. And look at the trend: raising the mean wind speed by half (5.3 → 8.0 m/s) multiplies the capacity factor by about 2.5 (15 % → 38 %). **Site quality dominates wind economics** — which is why developers measure wind for a year or more before building.

### Solar: where is the sun?

PV output depends first on the angle of the sun. Its height is described by the **solar zenith angle** z (0° = straight overhead, 90° = horizon). From astronomy:

- **declination** δ: the sun's latitude, swinging between ±23.45° over the year
- **hour angle** ω: 15° per hour away from solar noon, corrected by the **equation of time** (the sun runs up to ±16 minutes ahead of or behind the clock)
- **cos z = sin φ sin δ + cos φ cos δ cos ω**, with φ the site latitude

The function below uses Spencer's series for δ and the equation of time. Then **validate** it against pvlib, the standard Python library for solar modelling, for every hour of a year in Split.

In [7]:
def solar_zenith_deg(times_utc: pd.DatetimeIndex, lat: float, lon: float) -> np.ndarray:
    """Solar zenith angle (degrees) from Spencer's declination and equation of time."""
    doy = times_utc.dayofyear.to_numpy()
    hour = times_utc.hour.to_numpy() + times_utc.minute.to_numpy() / 60
    g = 2 * np.pi / 365 * (doy - 1 + (hour - 12) / 24)
    decl = (0.006918 - 0.399912 * np.cos(g) + 0.070257 * np.sin(g) - 0.006758 * np.cos(2 * g)
            + 0.000907 * np.sin(2 * g) - 0.002697 * np.cos(3 * g) + 0.00148 * np.sin(3 * g))
    eot_min = 229.18 * (0.000075 + 0.001868 * np.cos(g) - 0.032077 * np.sin(g)
                        - 0.014615 * np.cos(2 * g) - 0.040849 * np.sin(2 * g))
    solar_time_min = hour * 60 + eot_min + 4 * lon
    omega = np.radians(solar_time_min / 4 - 180)
    phi = np.radians(lat)
    cos_z = np.sin(phi) * np.sin(decl) + np.cos(phi) * np.cos(decl) * np.cos(omega)
    return np.degrees(np.arccos(np.clip(cos_z, -1, 1)))

lat, lon = 43.51, 16.44                                    # Split
times = pd.date_range("2025-01-01 00:30", periods=8760, freq="h", tz="UTC")   # mid-hour
zen_ours = solar_zenith_deg(times, lat, lon)
zen_pvlib = pvlib.solarposition.get_solarposition(times, lat, lon)["zenith"].to_numpy()
sun_up = zen_pvlib < 85
error = np.abs(zen_ours - zen_pvlib)[sun_up]
assert error.max() < 1.0, "solar position more than 1 degree off"
print(f"zenith error vs pvlib while the sun is up: max {error.max():.2f} deg, mean {error.mean():.2f} deg")

zenith error vs pvlib while the sun is up: max 0.55 deg, mean 0.28 deg


Our hand-written formula stays within about half a degree of pvlib all year — accurate enough for energy modelling, and now you know exactly what is inside. (pvlib uses a more precise astronomical algorithm; for real projects, use it.) Note the **mid-hour** timestamps: an hourly energy value describes a whole hour, so evaluate the sun in its middle.

### Clear-sky irradiance and PV output

Sunlight on a horizontal surface is the **global horizontal irradiance** (GHI, W/m²). On a cloudless day it depends mainly on the zenith angle. The simple **Haurwitz** clear-sky model:

**GHI_clear = 1098 · cos z · exp(−0.059 / cos z)**  (W/m², for the sun above the horizon)

A PV array's output relative to its rating is roughly **GHI / 1000 W/m² × performance ratio**, where the performance ratio (here 0.86) covers inverter, wiring, soiling and temperature losses. This treats panels as horizontal — a deliberate simplification; tilted panels collect more.

In [8]:
cos_z = np.clip(np.cos(np.radians(zen_ours)), 0, None)
ghi_clear = np.where(cos_z > 0.01, 1098 * cos_z * np.exp(-0.059 / np.maximum(cos_z, 0.01)), 0.0)
ghi_pvlib = pvlib.clearsky.haurwitz(pd.Series(zen_ours, index=times))["ghi"].to_numpy()
assert np.nanmax(np.abs(ghi_clear - ghi_pvlib)) < 0.1, "Haurwitz implementation differs from pvlib"

pv_pu = np.clip(ghi_clear / 1000 * 0.86, 0, 1)
monthly = pd.Series(pv_pu, index=times).groupby(times.month).mean()
print(f"annual clear-sky irradiation: {ghi_clear.sum() / 1000:,.0f} kWh/m2")
print(f"clear-sky PV capacity factor (horizontal, PR 0.86): {pv_pu.mean():.3f}")
monthly.round(3).rename("clear-sky CF").rename_axis("month").to_frame().T

annual clear-sky irradiation: 1,984 kWh/m2
clear-sky PV capacity factor (horizontal, PR 0.86): 0.195


month            1      2      3      4      5      6      7      8     9   \
clear-sky CF  0.083  0.124  0.182  0.244  0.291  0.312  0.304  0.266  0.21   

month            10     11     12  
clear-sky CF  0.147  0.097  0.073  

Even with a cloudless sky every day of the year, horizontal PV in Split reaches a capacity factor of only about 19 %: half the hours are night, and the sun is low in winter (compare January and July). **Real** capacity factors are lower still, because clouds, haze and dust reduce irradiance. This clear-sky value is therefore an **upper bound** — a useful check: any PV capacity factor above it for this site is a data or unit error.

### Real weather: Open-Meteo (ERA5 reanalysis)

For real profiles you need measured or reanalysed weather. [Open-Meteo](https://open-meteo.com/en/docs/historical-weather-api) offers hourly **ERA5 reanalysis** data (a consistent global reconstruction of past weather) back to 1940, free for non-commercial use and without an API key (CC BY 4.0 — credit the source). Two details matter:

- wind speed defaults to **km/h** — request `wind_speed_unit=ms`, or every wind result will be wrong by a factor of 3.6;
- `shortwave_radiation` (GHI) is the **mean of the preceding hour**: the value stamped 12:00 describes 11:00–12:00.

Other good sources: [Renewables.ninja](https://www.renewables.ninja/) (ready-made capacity factor profiles) and [PVGIS](https://joint-research-centre.ec.europa.eu/photovoltaic-geographical-information-system-pvgis_en) (EU solar data). The cell below needs internet access and was not executed when this notebook was prepared — run it on your machine.

In [9]:
def fetch_open_meteo(lat: float, lon: float, start: str, end: str) -> pd.DataFrame:
    """Hourly ERA5 wind speed at 100 m (m/s) and GHI (W/m2, preceding-hour mean) from Open-Meteo."""
    import requests
    params = {"latitude": lat, "longitude": lon, "start_date": start, "end_date": end,
              "hourly": "wind_speed_100m,shortwave_radiation", "wind_speed_unit": "ms", "timezone": "UTC"}
    r = requests.get("https://archive-api.open-meteo.com/v1/archive", params=params, timeout=60)
    r.raise_for_status()
    h = r.json()["hourly"]
    return pd.DataFrame({"wind_100m_m_per_s": h["wind_speed_100m"], "ghi_W_per_m2": h["shortwave_radiation"]},
                        index=pd.to_datetime(h["time"], utc=True))

try:
    weather = fetch_open_meteo(lat, lon, "2024-01-01", "2024-12-31")
    print(weather.describe().round(1))
except Exception as exc:                       # no internet, rate limit, ...
    weather = None
    print(f"Skipped (no data downloaded): {type(exc).__name__}")

Skipped (no data downloaded): ConnectionError


With real data the pipeline is: wind speed at 100 m → hub height → `power_curve` → wind `p_max_pu`; GHI → `/1000 × PR` → solar `p_max_pu`. Those two series replace the synthetic ones in your M1.2 network — you will do that in the exercise.

## Check your understanding

**Wind**

1. A site's mean wind speed is 10 % higher than expected. Using the cube law, estimate the change in power in the wind. Why is the change in capacity factor smaller than that?
2. Why does a turbine stop producing above 25 m/s, and why does the power curve go flat above 12 m/s?

**Solar**

1. Before running anything: in which month is the clear-sky capacity factor highest, and roughly how much higher than in December? Check with `monthly`.
2. A dataset reports a 23 % annual capacity factor for horizontal PV near Split. Is that plausible? What would you check first?

**Data**

1. What goes wrong if you forget `wind_speed_unit=ms` but your power curve expects m/s? Would the solver notice?
2. Why does it matter that GHI is a preceding-hour mean when you combine it with hourly load from ENTSO-E (M1.1)?